In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt

RICOTTA_LOG = 'runs/ricotta-2.0/r1-wsd10(minlr0.1)/train.log'
GRUYERE_LOG = 'runs/gruyere-2.0/r1/gruyere_2_0/train.log'
GOUDA_LOG = 'runs/gouda-2.0/r1/gouda_2_0/train.log'

LOGS = {
    "Ricotta-2.0": RICOTTA_LOG,
    "Gruyère-2.0": GRUYERE_LOG,
    "Gouda-2.0": GOUDA_LOG,
}


def parse_log(log_file):
    train_rows = []
    val_rows = []

    with open(log_file, "r") as f:
        for line in f:
            parts = line.split()

            if len(parts) < 2 or not parts[0].isdigit():
                continue

            step = int(parts[0])
            kind = parts[1]

            values = {
                k: float(v)
                for k, v in re.findall(
                    r"(\w+)=([\d.eE+-]+)",
                    line
                )
            }

            if kind == "train":
                train_rows.append({
                    "step": step,
                    **values
                })

            elif kind == "val":
                val_rows.append({
                    "step": step,
                    **values
                })

    train = pd.DataFrame(train_rows)
    val = pd.DataFrame(val_rows)

    return train, val


# Parse all logs
data = {
    name: parse_log(path)
    for name, path in LOGS.items()
}

FileNotFoundError: [Errno 2] No such file or directory: 'runs/ricotta-2.0/r1-wsd10(minlr0.1)/train.log'

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(12, 6))

for name, (train, val) in data.items():

    smooth = train["loss"].rolling(
        100,
        min_periods=1
    ).mean()

    line, = ax.plot(
        train["tokens"] / 1e9,
        smooth,
        linewidth=2,
        label=f"{name} train",
    )

    if len(val):
        ax.plot(
            val["tokens"] / 1e9,
            val["loss"],
            marker="o",
            markersize=4,
            linewidth=1.5,
            linestyle="--",
            color=line.get_color(),
            label=f"{name} val",
        )


ax.set_xlabel("Training Tokens (billions)")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Gouda Architecture 2.0 — Loss vs Training Tokens")

ax.set_ylim(
    bottom=3,
    top=4,
)

ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()